# Retail FAQ Chatbot

## Objective

The objective of this notebook is to build an intelligent FAQ chatbot for a retail clothing store.

The chatbot will:

- Understand customer questions
- Predict the customer's intent
- Return an appropriate response
- Save the trained chatbot model for deployment

This chatbot will later be integrated with the FastAPI backend.

In [78]:
print("Hello")

Hello


In [79]:
!pip install scikit-learn joblib nltk


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [80]:
import json
import random
import joblib

import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split

from sklearn.metrics import classification_report

## Load Intent Dataset

In [81]:
with open("../data/intents.json","r") as file:
    intents = json.load(file)

In [82]:
# dict_keys(['intents'])

In [83]:
intents.keys()

dict_keys(['intents'])

In [84]:
len(intents["intents"])

16

## Convert JSON Data into a DataFrame

The chatbot dataset is currently stored in JSON format.

To train a Machine Learning model, we convert it into a tabular format where:

- Each user query (pattern) becomes one training sample.
- Each intent tag becomes the target label.

In [85]:
patterns = []
tags = []

for intent in intents["intents"]:

    tag = intent["tag"]

    for pattern in intent["patterns"]:

        patterns.append(pattern)
        tags.append(tag)

In [86]:
df = pd.DataFrame({
    "Pattern": patterns,
    "Tag": tags
})

df.head()

,Pattern,Tag
0,hi,greeting
1,hello,greeting
2,hey,greeting
3,good morning,greeting
4,good evening,greeting


In [87]:
df.sample(10, random_state=42)

,Pattern,Tag
167,package status,order_tracking
230,size small,size_guide
25,bye bye,goodbye
63,credit card,payment
9,hi team,greeting
110,product return,return_policy
186,cancel item,cancel_order
143,replacement,exchange
244,sale,discounts
224,fit guide,size_guide


In [88]:
print("Dataset Shape :", df.shape)

print("\n")

print(df["Tag"].value_counts())

Dataset Shape : (320, 2)


Tag
greeting                20
goodbye                 20
thanks                  20
payment                 20
shipping                20
return_policy           20
refund                  20
exchange                20
order_tracking          20
cancel_order            20
product_availability    20
size_guide              20
discounts               20
contact_support         20
store_hours             20
unknown                 20
Name: count, dtype: int64


## Check for Missing Values

Before training, we verify that there are no missing values in the dataset.

In [89]:
df.isnull().sum()

Pattern    0
Tag        0
dtype: int64

## Split Dataset

The dataset is divided into:

- Training Set (80%)
- Testing Set (20%)

The `stratify` parameter ensures that each intent is represented proportionally in both sets.

In [90]:
X_train, X_test, y_train, y_test = train_test_split(
    df["Pattern"],
    df["Tag"],
    test_size=0.20,
    random_state=42,
    stratify=df["Tag"]
)

In [91]:
print("Training Samples :", len(X_train))
print("Testing Samples  :", len(X_test))

Training Samples : 256
Testing Samples  : 64


## TF-IDF Vectorization

Convert the text patterns into numerical vectors using TF-IDF.
These vectors will be used to train the Logistic Regression model.

In [92]:
vectorizer = TfidfVectorizer()

In [93]:
X_train = vectorizer.fit_transform(X_train)

X_test = vectorizer.transform(X_test)

In [94]:
print("Training Shape :", X_train.shape)

print("Testing Shape :", X_test.shape)

Training Shape : (256, 241)
Testing Shape : (64, 241)


## Train Logistic Regression Model

Train the chatbot to classify the intent of user queries.

In [95]:
chatbot_model = LogisticRegression(
    random_state=42
)

In [96]:
chatbot_model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solve

## Model Evaluation

In [97]:
predictions = chatbot_model.predict(X_test)

In [98]:
print(classification_report(y_test, predictions))

                      precision    recall  f1-score   support

        cancel_order       0.80      1.00      0.89         4
     contact_support       0.75      0.75      0.75         4
           discounts       1.00      0.75      0.86         4
            exchange       1.00      0.75      0.86         4
             goodbye       0.67      0.50      0.57         4
            greeting       1.00      0.50      0.67         4
      order_tracking       0.60      0.75      0.67         4
             payment       1.00      0.50      0.67         4
product_availability       0.80      1.00      0.89         4
              refund       1.00      1.00      1.00         4
       return_policy       1.00      1.00      1.00         4
            shipping       0.75      0.75      0.75         4
          size_guide       1.00      0.75      0.86         4
         store_hours       0.75      0.75      0.75         4
              thanks       1.00      0.75      0.86         4
       

In [99]:
accuracy = chatbot_model.score(X_test, y_test)

print(f"Accuracy : {accuracy:.2f}")

Accuracy : 0.77


## Save Model

In [100]:
import os

os.makedirs("../models", exist_ok=True)

In [101]:
joblib.dump(chatbot_model, "../models/chatbot_model.pkl")

joblib.dump(vectorizer, "../models/chatbot_vectorizer.pkl")

['../models/chatbot_vectorizer.pkl']

In [102]:
print("Chatbot Model Saved Successfully!")

Chatbot Model Saved Successfully!


## Test the Chatbot

Load the trained chatbot model and test it using custom customer queries.

In [103]:
loaded_chatbot = joblib.load("../models/chatbot_model.pkl")

loaded_vectorizer = joblib.load("../models/chatbot_vectorizer.pkl")

In [104]:
def predict_intent(text):

    vector = loaded_vectorizer.transform([text])

    prediction = loaded_chatbot.predict(vector)

    return prediction[0]

In [105]:
questions = [
    "Hi",
    "Can I return my order?",
    "How can I pay?",
    "When will my order arrive?",
    "Bye"
]

for question in questions:

    print("Question :", question)

    print("Intent   :", predict_intent(question))

    print("-"*50)

Question : Hi
Intent   : greeting
--------------------------------------------------
Question : Can I return my order?
Intent   : return_policy
--------------------------------------------------
Question : How can I pay?
Intent   : payment
--------------------------------------------------
Question : When will my order arrive?
Intent   : order_tracking
--------------------------------------------------
Question : Bye
Intent   : goodbye
--------------------------------------------------


In [106]:
def chatbot_response(user_input):

    intent = predict_intent(user_input)

    for item in intents["intents"]:

        if item["tag"] == intent:

            return random.choice(item["responses"])

In [107]:
while True:

    user = input("You : ")

    if user.lower() == "exit":

        print("Bot : Goodbye!")

        break

    print("Bot :", chatbot_response(user))

Bot : Welcome! What can I do for you today?
Bot : Our customer support is happy to assist you.
Bot : We're here to help with your queries.
Bot : We'll notify you once the refund is processed.
Bot : Shipping details are shown during checkout.
Bot : A tracking link is shared after dispatch.
Bot : Tracking information becomes available after dispatch.
Bot : Please ask me about orders, payments, shipping or returns.
Bot : I'm designed to answer retail-related questions.
Bot : Shipping details are shown during checkout.
Bot : Can you rephrase your question?
Bot : Our return policy allows eligible returns within 30 days.
Bot : I couldn't identify your request.
Bot : Goodbye!
